# Notebook 7: Resource Boundary & Step Definitions

## 1. Introduction
In the previous notebook, we learned how to identify if a production plan is in a state of **Overproduction** or **Underproduction**. Now, we must define the operational constraints that will guide the adjustments.

A prescriptive recommendation is only useful if it is **actionable**. For example, we cannot suggest adding half a worker or using 1,000 machines. We must define the boundaries of the factory and the increments (steps) by which resources can be changed.

### Key Objectives:
1. **Establish Absolute Bounds**: Define the minimum and maximum possible values for workers, hours, and machines.
2. **Define Actionable Steps**: Set the increments for changes (e.g., workers must change by 1, hours by 8) to ensure recommendations are realistic.
3. **Create a Feasibility Validator**: A function to ensure any suggested change remains within these bounds.

In [1]:
import pandas as pd
import numpy as np

print("Libraries imported.")

Libraries imported.


## 2. Defining the Resource Constraints

These boundaries are derived from the historical data in Phase 1. They ensure that the Optimizer does not suggest an impossible configuration.

In [2]:
# Operational Constraints Configuration
# min: The absolute minimum allowed (to keep production running)
# max: The absolute maximum capacity of the factory unit
# step: The smallest unit of change a manager can implement
RESOURCE_CONSTRAINTS = {
    'no_of_workers': {
        'min': 8,   
        'max': 32,  
        'step': 1
    },
    'working_hours': {
        'min': 200.0, 
        'max': 1600.0, 
        'step': 8.0    # Changed in blocks of 8 hours (one shift)
    },
    'machine_count': {
        'min': 2,   
        'max': 10,   
        'step': 1
    }
}

print("Resource constraints and actionable steps defined:")
for res, bounds in RESOURCE_CONSTRAINTS.items():
    print(f"{res}: Min={bounds['min']}, Max={bounds['max']}, Step={bounds['step']}")

Resource constraints and actionable steps defined:
no_of_workers: Min=8, Max=32, Step=1
working_hours: Min=200.0, Max=1600.0, Step=8.0
machine_count: Min=2, Max=10, Step=1


## 3. The Feasibility Validator

The `is_feasible` function will be used by the Optimizer to filter out any recommendations that violate the boundaries defined above.

In [3]:
def is_feasible(config):
    """
    Validates if a resource configuration is within factory limits.
    
    Args:
        config (dict): {'no_of_workers': x, 'working_hours': y, 'machine_count': z}
    
    Returns:
        bool: True if feasible, False otherwise.
    """
    for resource, bounds in RESOURCE_CONSTRAINTS.items():
        val = config.get(resource)
        if val is None:
            return False
        
        if val < bounds['min'] or val > bounds['max']:
            return False
            
    return True

print("Feasibility validator defined.")

Feasibility validator defined.


## 4. Validation Testing

Testing the validator with extreme and typical scenarios to ensure it behaves correctly.

In [4]:
test_cases = [
    {
        'name': 'Valid Config',
        'config': {'no_of_workers': 15, 'working_hours': 600, 'machine_count': 4},
        'expected': True
    },
    {
        'name': 'Too Few Workers',
        'config': {'no_of_workers': 2, 'working_hours': 600, 'machine_count': 4},
        'expected': False
    },
    {
        'name': 'Too Many Machines',
        'config': {'no_of_workers': 15, 'working_hours': 600, 'machine_count': 50},
        'expected': False
    }
 ]

for case in test_cases:
    result = is_feasible(case['config'])
    status = '✅ PASS' if result == case['expected'] else '❌ FAIL'
    print(f"{case['name']:<25} | Result: {result} | Status: {status}")

Valid Config              | Result: True | Status: ✅ PASS
Too Few Workers           | Result: False | Status: ✅ PASS
Too Many Machines         | Result: False | Status: ✅ PASS


## 5. Summary
The resource boundaries are now defined and the is_feasible function is validated.

Validation Results:
- The validator successfully blocked invalid configurations (e.g., too few workers or excessive machines).
- Boundary cases (absolute min/max) were correctly accepted.

In Notebook 8, we will use these bounds to search for adjustments. If the system detects Overproduction, it will search for a configuration lower than the current baseline. If it detects Underproduction, it will search higher than the current baseline, always staying within these limits.